In [33]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import json
from pathlib import Path

import pandas as pd

from datasmith.docker.context import ContextRegistry, DockerContext, Task
from datasmith.notebooks.utils import update_cr

/mnt/sdd1/atharvas/formulacode/datasmith


In [48]:
# # Original
# verified_registry_pth = Path("/mnt/sdd1/atharvas/formulacode/datasmith/scratch/context_registry_final_filtered.json")
# verified_repos_pth = Path(
#     "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/aws_ecr_filtered_formulacode-verified.parquet"
# )
# valid_tasks = None # Load all tasks
# registry_path = Path("scratch/formulacode_verified_context_registry.json")
# dataset_path = Path("dataset/formulacode_verified")

# For new dataset
verified_registry_pth = Path("/mnt/sdd1/atharvas/formulacode/datasmith/scratch/context_registry_final_filtered.json")
verified_repos_pth = Path(
    "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/filtered_formulacode-new.parquet"
)

valid_tasks = list(
    map(
        eval,
        json.loads(
            Path(
                "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/valid_tasks_new_200.json"
            ).read_text()
        ).keys(),
    )
)

registry_path = Path("scratch/formulacode_verified_new_context_registry.json")
dataset_path = Path("dataset_new/formulacode_verified")

In [49]:
verified_repos = pd.read_parquet(verified_repos_pth)

if valid_tasks is not None:
    verified_repos = verified_repos[
        verified_repos[["repo_name", "pr_merge_commit_sha"]].apply(tuple, axis=1).isin(valid_tasks)
    ]

registry = update_cr(ContextRegistry.load_from_file(verified_registry_pth))

In [ ]:
def get_task(repo_name, base_commit_sha):
    for task in registry.registry:
        repo_name = f"{task.owner}/{task.repo}"
        sha = task.sha
        if repo_name == repo_name and sha == base_commit_sha:
            return task, registry.registry[task]
    return None, None


new_registry = ContextRegistry() if not registry_path.exists() else ContextRegistry.load_from_file(registry_path)

print(len(new_registry.registry))
verified_repos["is_available"] = False
for idx, row in verified_repos.iterrows():
    repo_name = row["repo_name"]
    base_commit_sha = row["pr_base"]["sha"]
    task, context = get_task(repo_name, base_commit_sha)
    if task in new_registry.registry:
        print(f"Skipping {task} as already in registry")
        verified_repos.at[idx, "is_available"] = True
        continue
    if task is not None:
        verified_repos.at[idx, "is_available"] = True
        new_registry.register(task.with_tag("pkg"), context)


print(len(new_registry.registry))
new_registry.save_to_file(registry_path)

17:03:23 INFO     datasmith.docker.context: Context registry saved to scratch/formulacode_verified_new_context_registry.json


1
Skipping Task(owner='scikit-image', repo='scikit-image', sha='487a3668f9d83686c5081d609c5f8fdc279fa2aa', commit_date=1636931684.0, env_payload='{"dependencies": ["alabaster==0.7.16", "astropy==6.1.7", "astropy-iers-data==0.2025.11.3.0.38.37", "asttokens==3.0.0", "asv==0.6.5", "asv-runner==0.2.1", "babel==2.17.0", "build==1.3.0", "certifi==2025.10.5", "charset-normalizer==3.4.4", "choreographer==1.2.0", "click==8.3.0", "cloudpickle==3.1.2", "codecov==2.1.13", "comm==0.2.3", "contourpy==1.3.2", "coverage==7.11.0", "cycler==0.12.1", "dask==2025.10.0", "decorator==5.2.1", "distlib==0.4.0", "docutils==0.17.1", "exceptiongroup==1.3.0", "executing==2.2.1", "filelock==3.20.0", "flake8==7.3.0", "fonttools==4.60.1", "fsspec==2025.10.0", "idna==3.11", "imageio==2.37.0", "imagesize==1.4.1", "importlib-metadata==8.7.0", "iniconfig==2.3.0", "ipython==8.37.0", "ipywidgets==8.1.8", "jedi==0.19.2", "jinja2==3.1.6", "joblib==1.5.2", "json5==0.12.1", "jupyterlab-widgets==3.0.16", "kaleido==1.1.0", "kiw

In [52]:
# save each dockerfile context in a folder called formulacode_verified/{repo_name}/{sha}/{all files}
def save_context(context: DockerContext, task: Task, task_dir: Path):
    task_dir.mkdir(parents=True, exist_ok=True)
    task_dir.joinpath("Dockerfile").write_text(context.dockerfile_data)
    task_dir.joinpath("entrypoint.sh").write_text(context.entrypoint_data)
    task_dir.joinpath("docker_build_base.sh").write_text(context.base_building_data)
    task_dir.joinpath("docker_build_run.sh").write_text(context.run_building_data)
    task_dir.joinpath("docker_build_env.sh").write_text(context.env_building_data)
    task_dir.joinpath("docker_build_final.sh").write_text(context.final_building_data)
    task_dir.joinpath("docker_build_pkg.sh").write_text(context.building_data)
    task_dir.joinpath("profile.sh").write_text(context.profile_data)
    task_dir.joinpath("run_tests.sh").write_text(context.run_tests_data)
    task_dir.joinpath("task.txt").write_text(repr(task))


for _, row in verified_repos.iterrows():
    task_id = row["task_id"]
    repo_name = row["repo_name"]
    base_commit_sha = row["pr_base"]["sha"]
    task, context = get_task(repo_name, base_commit_sha)
    if not task or not context:
        print(f"Skipping {task_id} as context not found")
        continue
    task_dir = dataset_path / repo_name.replace("/", "_") / base_commit_sha
    save_context(context, task, task_dir)

Skipping pydata_xarray_7 as context not found


In [53]:
dataset_path.parent.joinpath("config.json").write_text(
    json.dumps({"context_registry_path": str(registry_path), "dockerhub_repo": "all", "dockerhub_namespace": None})
)

137